In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from geopy.distance import geodesic
from itertools import product
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('All imports OK')

All imports OK


## 1. Load & Clean Data (Jan + Feb + Mar 2025)

In [2]:
# ── Load all 3 months ─────────────────────────────────────────────────────────
files = [
    'yellow_tripdata_2025-01.parquet',
    'yellow_tripdata_2025-02.parquet',
    'yellow_tripdata_2025-03.parquet'
]

dfs = []
for f in files:
    tmp = pd.read_parquet(f, columns=[
        'tpep_pickup_datetime', 'PULocationID',
        'trip_distance', 'fare_amount', 'total_amount'
    ])
    dfs.append(tmp)
    print(f'{f}: {tmp.shape[0]:,} rows')

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal raw rows: {df.shape[0]:,}')

yellow_tripdata_2025-01.parquet: 3,475,226 rows
yellow_tripdata_2025-02.parquet: 3,577,543 rows
yellow_tripdata_2025-03.parquet: 4,145,257 rows

Total raw rows: 11,198,026


In [3]:
# ── Clean ─────────────────────────────────────────────────────────────────────
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

df = df[
    (df['fare_amount']    > 0)   &
    (df['fare_amount']    < 500) &
    (df['trip_distance']  > 0)   &
    (df['trip_distance']  < 120) &
    (df['PULocationID'].between(1, 263))
]

df = df.dropna(subset=['tpep_pickup_datetime', 'PULocationID', 'fare_amount'])
print(f'After cleaning: {df.shape[0]:,} rows')
print(f'Unique zones  : {df["PULocationID"].nunique()}')

After cleaning: 10,389,458 rows
Unique zones  : 258


## 2. Zone Distribution Check (Reviewer Fix #1)

In [4]:
# ── Zone trip distribution ────────────────────────────────────────────────────
zone_counts = df['PULocationID'].value_counts()
print(f'Top 10 zones by trip count:')
print(zone_counts.head(10))

# Gini coefficient — measures spatial imbalance (0=perfect, 1=one zone all)
def gini(arr):
    arr = np.sort(arr.values.astype(float))
    n   = len(arr)
    return (2 * np.sum(np.arange(1, n+1) * arr) / (n * arr.sum())) - (n+1)/n

g = gini(zone_counts)
print(f'\nGini coefficient (spatial imbalance): {g:.4f}')
print('(0 = perfectly balanced, 1 = all trips in one zone)')

# Entropy
probs   = zone_counts / zone_counts.sum()
entropy = -(probs * np.log2(probs + 1e-10)).sum()
print(f'Spatial entropy: {entropy:.4f} bits  (max = {np.log2(len(zone_counts)):.2f})')

Top 10 zones by trip count:
PULocationID
161    487185
237    473418
236    437134
132    398732
230    353386
186    349137
162    343882
142    316535
234    294034
170    283844
Name: count, dtype: int64

Gini coefficient (spatial imbalance): 0.8339
(0 = perfectly balanced, 1 = all trips in one zone)
Spatial entropy: 5.8257 bits  (max = 8.01)


In [5]:
# ── Zone distribution histogram ───────────────────────────────────────────────
plt.figure(figsize=(10, 4))
zone_counts.head(50).plot(kind='bar', color='steelblue', edgecolor='black', linewidth=0.4)
plt.xlabel('PULocationID (top 50 zones)')
plt.ylabel('Trip Count')
plt.title('Zone Trip Distribution — Top 50 Zones\n(Using official NYC 263-zone system)')
plt.xticks(fontsize=6, rotation=90)
plt.tight_layout()
plt.savefig('zone_distribution.png', dpi=150)
plt.show()
print('Saved zone_distribution.png')

Saved zone_distribution.png


## 3. Temporal Demand Aggregation

In [7]:
# ── 15-min aggregation ────────────────────────────────────────────────────────
df['time_window'] = df['tpep_pickup_datetime'].dt.floor('15min')

demand_df = df.groupby(['PULocationID', 'time_window']).agg(
    trip_count=('fare_amount', 'count'),
    avg_fare  =('fare_amount', 'mean')
).reset_index()

# ── Driver supply estimate: active drivers per zone per window ────────────────
# Proxy: unique trips as proxy for driver-minutes (conservative estimate)
# We estimate ~3 trips/driver/hour → 0.75 trips per 15-min window per driver
TRIPS_PER_DRIVER_PER_WINDOW = 0.75
demand_df['est_drivers'] = np.ceil(
    demand_df['trip_count'] / TRIPS_PER_DRIVER_PER_WINDOW
).clip(lower=1)

# ── Per-driver revenue (CORRECTED earnings — Reviewer Fix #2) ─────────────────
demand_df['per_driver_revenue'] = (
    demand_df['trip_count'] * demand_df['avg_fare']
) / demand_df['est_drivers']

print(f'Demand records: {demand_df.shape[0]:,}')
print(f'\nSample:')
print(demand_df.head(5).to_string(index=False))

Demand records: 786,579

Sample:
 PULocationID         time_window  trip_count  avg_fare  est_drivers  per_driver_revenue
            1 2025-01-01 06:00:00           1      99.0          2.0                49.5
            1 2025-01-01 13:00:00           1      37.0          2.0                18.5
            1 2025-01-01 14:00:00           1      18.4          2.0                 9.2
            1 2025-01-01 16:30:00           1     105.0          2.0                52.5
            1 2025-01-01 18:45:00           1      24.0          2.0                12.0


## 4. Feature Engineering

In [8]:
# ── Time features ─────────────────────────────────────────────────────────────
demand_df['hour']    = demand_df['time_window'].dt.hour
demand_df['weekday'] = demand_df['time_window'].dt.weekday
demand_df['month']   = demand_df['time_window'].dt.month

# Cyclical encoding
demand_df['hour_sin']    = np.sin(2 * np.pi * demand_df['hour']    / 24)
demand_df['hour_cos']    = np.cos(2 * np.pi * demand_df['hour']    / 24)
demand_df['weekday_sin'] = np.sin(2 * np.pi * demand_df['weekday'] / 7)
demand_df['weekday_cos'] = np.cos(2 * np.pi * demand_df['weekday'] / 7)
demand_df['month_sin']   = np.sin(2 * np.pi * demand_df['month']   / 12)
demand_df['month_cos']   = np.cos(2 * np.pi * demand_df['month']   / 12)

print('Time features done')

Time features done


In [10]:
# ── Lag & rolling features ────────────────────────────────────────────────────
demand_df = demand_df.sort_values(['PULocationID', 'time_window'])

for lag in [1, 2, 4, 8, 96]:          # 96 = 24 hrs ago (same time yesterday)
    demand_df[f'lag_{lag}'] = (
        demand_df.groupby('PULocationID')['trip_count'].shift(lag)
    )

demand_df['rolling_mean_4']  = (
    demand_df.groupby('PULocationID')['trip_count']
    .transform(lambda x: x.shift(1).rolling(4).mean())
)
demand_df['rolling_std_4']   = (
    demand_df.groupby('PULocationID')['trip_count']
    .transform(lambda x: x.shift(1).rolling(4).std())
)
demand_df['rolling_mean_96'] = (
    demand_df.groupby('PULocationID')['trip_count']
    .transform(lambda x: x.shift(1).rolling(96).mean())
)

demand_df = demand_df.dropna()
print(f'After lag/rolling dropna: {demand_df.shape[0]:,} rows')

After lag/rolling dropna: 741,726 rows


## 5. Train / Test Split (Temporal — No Leakage)

In [15]:
# ── Strict temporal split AFTER feature computation (Reviewer Fix #5) ─────────
split_date = demand_df['time_window'].quantile(0.8)
print(f'Train up to : {split_date}')
print(f'Test  from  : {split_date}')

train = demand_df[demand_df['time_window'] <= split_date]
test  = demand_df[demand_df['time_window'] >  split_date]

features = [
    'PULocationID',
    'hour_sin', 'hour_cos',
    'weekday_sin', 'weekday_cos',
    'month_sin', 'month_cos',
    'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_96',
    'rolling_mean_4', 'rolling_std_4', 'rolling_mean_96'
]

X_train, y_train = train[features], train['trip_count']
X_test,  y_test  = test[features],  test['trip_count']

print(f'Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}')

Train up to : 2025-03-16 22:00:00
Test  from  : 2025-03-16 22:00:00
Train size: 593,419 | Test size: 148,307


## 6. Train XGBoost Model

In [16]:
# ── Train ─────────────────────────────────────────────────────────────────────
model = xgb.XGBRegressor(
    objective      = 'reg:squarederror',
    n_estimators   = 800,
    learning_rate  = 0.02,
    max_depth      = 8,
    subsample      = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    random_state   = 42,
    n_jobs         = -1
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)
print('Training complete')

[0]	validation_0-rmse:20.46282
[100]	validation_0-rmse:5.93621
[200]	validation_0-rmse:5.19519
[300]	validation_0-rmse:5.12325
[400]	validation_0-rmse:5.10401
[500]	validation_0-rmse:5.09349
[600]	validation_0-rmse:5.08419
[700]	validation_0-rmse:5.07432
[799]	validation_0-rmse:5.06950
Training complete


## 7. Evaluation — Full Metrics (Reviewer Fix #6)

In [18]:
# ── Predict ───────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 0, None)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

# MAPE — reviewer requested
mask = y_test > 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

# Normalized RMSE
nrmse = rmse / (y_test.max() - y_test.min()) * 100

# Per-zone R2
test_res = test[['PULocationID', 'time_window']].copy()
test_res['actual']    = y_test.values
test_res['predicted'] = y_pred

print('=' * 45)
print('        MODEL EVALUATION METRICS')
print('=' * 45)
print(f'  MAE              : {mae:.4f}')
print(f'  RMSE             : {rmse:.4f}')
print(f'  R²               : {r2:.4f}')
print(f'  MAPE             : {mape:.2f}%')
print(f'  Normalized RMSE  : {nrmse:.2f}%')
print(f'  Demand range     : {int(y_test.min())} – {int(y_test.max())}')
print('=' * 45)

        MODEL EVALUATION METRICS
  MAE              : 2.5953
  RMSE             : 5.0695
  R²               : 0.9405
  MAPE             : 40.97%
  Normalized RMSE  : 1.96%
  Demand range     : 1 – 259


## 8. WSM Setup with Driver Supply (Reviewer Fix #3)

In [19]:
# ── Zone stats from test set ──────────────────────────────────────────────────
zone_stats = test_res.groupby('PULocationID').agg(
    predicted_demand = ('predicted', 'mean'),
    actual_demand    = ('actual',    'mean')
).reset_index()

# Average fare per zone
fare_by_zone = df.groupby('PULocationID')['fare_amount'].mean().reset_index()
fare_by_zone.columns = ['PULocationID', 'avg_fare']

# Demand density (trips per zone across all windows)
density = demand_df.groupby('PULocationID')['trip_count'].mean().reset_index()
density.columns = ['PULocationID', 'demand_density']

# Driver supply estimate per zone
supply = demand_df.groupby('PULocationID')['est_drivers'].mean().reset_index()
supply.columns = ['PULocationID', 'avg_drivers']

# Supply-demand ratio (lower = better for driver, less competition)
# Use NYC taxi zone centroids (approximate lat/lon per zone ID)
# We'll use a lookup for major zones; others get borough centroid
ZONE_COORDS = {
    # Key Manhattan zones
    161: (40.7589, -73.9851),  # Midtown Center
    162: (40.7614, -73.9776),  # Midtown East
    163: (40.7549, -73.9840),  # Midtown North
    164: (40.7484, -73.9967),  # Midtown South
    230: (40.7527, -73.9772),  # Times Sq/Theatre District
    237: (40.7736, -73.9566),  # Upper East Side N
    236: (40.7685, -73.9566),  # Upper East Side S
    186: (40.7282, -74.0776),  # Park Slope
    132: (40.6413, -73.7781),  # JFK Airport
    138: (40.7769, -73.8740),  # LaGuardia Airport
    229: (40.7549, -73.9840),  # Sutton Place/Turtle Bay N
    # Default fallback: Manhattan centroid
}
DEFAULT_COORD = (40.7580, -73.9855)  # Midtown Manhattan

zones = zone_stats \
    .merge(fare_by_zone, on='PULocationID', how='left') \
    .merge(density,      on='PULocationID', how='left') \
    .merge(supply,       on='PULocationID', how='left')

zones['lat'] = zones['PULocationID'].map(
    lambda z: ZONE_COORDS.get(z, DEFAULT_COORD)[0]
)
zones['lon'] = zones['PULocationID'].map(
    lambda z: ZONE_COORDS.get(z, DEFAULT_COORD)[1]
)

zones['predicted_demand'] = zones['predicted_demand'].clip(lower=0)

# Supply-demand ratio (Reviewer Fix #3)
zones['supply_demand_ratio'] = zones['avg_drivers'] / (zones['demand_density'] + 1e-6)

print(f'Zones available for WSM scoring: {len(zones)}')
print(zones[['PULocationID','predicted_demand','avg_fare',
             'demand_density','avg_drivers','supply_demand_ratio']]
      .sort_values('predicted_demand', ascending=False)
      .head(10).to_string(index=False))

Zones available for WSM scoring: 223
 PULocationID  predicted_demand  avg_fare  demand_density  avg_drivers  supply_demand_ratio
          161         60.186661 15.506349       58.794658    78.738299             1.339208
          237         55.038372 12.592348       58.183562    77.924408             1.339286
          132         52.853592 62.868205       46.862726    62.820507             1.340522
          236         49.638470 13.179939       55.455620    74.292614             1.339677
          230         43.821617 17.831469       41.330197    55.444669             1.341505
          162         43.102661 15.077839       41.802959    56.081628             1.341571
          186         42.742992 15.729092       41.220111    55.292519             1.341397
          138         42.464863 42.621077       38.174186    51.251999             1.342583
          142         38.666203 13.958263       39.098680    52.478944             1.342218
          234         37.429920 14.048512  

## 9. WSM Weight Grid Search (Reviewer Fix #8)

In [20]:
# ── Grid search over WSM weights ──────────────────────────────────────────────
# Objective: maximize per-driver revenue in recommended zone on validation set

FUEL_RATE  = 0.12   # USD per km  (fuel price / mileage: ~$3.5/gal ÷ 30mpg ≈ $0.12/km)
driver_loc = (40.7580, -73.9855)   # Driver starts at Midtown Manhattan

alpha_vals = [0.6, 0.8, 1.0]
beta_vals  = [0.2, 0.4, 0.6]
gamma_vals = [0.1, 0.2, 0.3]

best_score  = -np.inf
best_weights = (1.0, 0.5, 0.3)

def compute_wsm(driver_loc, zone_row, alpha, beta, gamma):
    zone_loc    = (zone_row['lat'], zone_row['lon'])
    distance    = geodesic(driver_loc, zone_loc).km
    # CORRECTED revenue = per-driver share (Reviewer Fix #2)
    revenue     = (zone_row['predicted_demand'] * zone_row['avg_fare']) / \
                  max(zone_row['avg_drivers'], 1)
    idle_pen    = 1 / (zone_row['demand_density'] + 1e-6)
    supply_pen  = zone_row['supply_demand_ratio']   # higher ratio = more competition
    fuel_cost   = distance * FUEL_RATE
    return alpha * revenue - beta * (idle_pen + supply_pen) - gamma * fuel_cost

for a, b, g in product(alpha_vals, beta_vals, gamma_vals):
    zones['wsm_tmp'] = zones.apply(
        lambda z: compute_wsm(driver_loc, z, a, b, g), axis=1
    )
    best_z   = zones.sort_values('wsm_tmp', ascending=False).iloc[0]
    # Metric: per-driver revenue of chosen zone
    score = best_z['predicted_demand'] * best_z['avg_fare'] / max(best_z['avg_drivers'], 1)
    if score > best_score:
        best_score   = score
        best_weights = (a, b, g)

alpha, beta, gamma = best_weights
print(f'Best WSM weights  →  α={alpha}, β={beta}, γ={gamma}')
print(f'Best per-driver revenue score: ${best_score:.2f}')

Best WSM weights  →  α=0.6, β=0.2, γ=0.1
Best per-driver revenue score: $52.89


## 10. Final WSM Scoring & Recommendation

In [21]:
# ── Compute final WSM scores ──────────────────────────────────────────────────
zones['WSM_score'] = zones.apply(
    lambda z: compute_wsm(driver_loc, z, alpha, beta, gamma), axis=1
)

zones_sorted = zones.sort_values('WSM_score', ascending=False)

print('\nTop 10 Zones by WSM Score:')
print(zones_sorted[['PULocationID','predicted_demand','avg_fare',
                     'avg_drivers','supply_demand_ratio','WSM_score']]
      .head(10).to_string(index=False))

best_zone = zones_sorted.iloc[0]
per_driver_rev = (best_zone['predicted_demand'] * best_zone['avg_fare']) / \
                  max(best_zone['avg_drivers'], 1)

print(f"\n{'='*52}")
print(f"           RECOMMENDED ZONE")
print(f"{'='*52}")
print(f"  Zone (PULocationID) : {int(best_zone['PULocationID'])}")
print(f"  Predicted Demand    : {best_zone['predicted_demand']:.2f} trips/15min")
print(f"  Avg Fare            : ${best_zone['avg_fare']:.2f}")
print(f"  Est. Drivers        : {best_zone['avg_drivers']:.1f}")
print(f"  Supply/Demand Ratio : {best_zone['supply_demand_ratio']:.4f}")
print(f"  Per-Driver Revenue  : ${per_driver_rev:.2f} per window")
print(f"  WSM Score           : {best_zone['WSM_score']:.4f}")
print(f"{'='*52}")


Top 10 Zones by WSM Score:
 PULocationID  predicted_demand  avg_fare  avg_drivers  supply_demand_ratio  WSM_score
          132         52.853592 62.868205    62.820507             1.340522  31.202274
            1          1.098869 75.896792     2.000000             1.999998  24.420181
          138         42.464863 42.621077    51.251999             1.342583  20.798729
           93          1.146190 61.766959     2.133525             1.882202  19.356884
           70          5.041017 41.700231     6.466892             1.413759  19.177012
          219          1.260791 49.725196     2.199666             1.842962  16.564543
           10          1.457184 47.099674     2.415048             1.735718  16.560424
          117          1.262475 45.577728     2.171079             1.857141  15.359467
          195          1.931332 40.222745     2.960972             1.645253  15.301301
          207          1.101985 48.315085     2.026316             1.974357  15.175566

           REC

## 11. Multiple Baseline Comparison (Reviewer Fix #4)

In [22]:
# ── Baseline 1: Random zone ───────────────────────────────────────────────────
random_rev = (
    zones['predicted_demand'] * zones['avg_fare'] / zones['avg_drivers']
).mean()

# ── Baseline 2: Nearest zone (closest to driver) ──────────────────────────────
zones['dist_km'] = zones.apply(
    lambda z: geodesic(driver_loc, (z['lat'], z['lon'])).km, axis=1
)
nearest_zone = zones.sort_values('dist_km').iloc[0]
nearest_rev  = (nearest_zone['predicted_demand'] * nearest_zone['avg_fare']) / \
                max(nearest_zone['avg_drivers'], 1)

# ── Baseline 3: Greedy demand (highest demand zone, ignores cost) ─────────────
greedy_zone = zones.sort_values('predicted_demand', ascending=False).iloc[0]
greedy_rev  = (greedy_zone['predicted_demand'] * greedy_zone['avg_fare']) / \
               max(greedy_zone['avg_drivers'], 1)

# ── Our system ────────────────────────────────────────────────────────────────
our_rev = per_driver_rev

print('='*55)
print('       BASELINE COMPARISON (Per-Driver Revenue)')
print('='*55)
print(f'  Random Zone              : ${random_rev:.2f} / 15-min window')
print(f'  Nearest Zone             : ${nearest_rev:.2f} / 15-min window')
print(f'  Greedy Demand            : ${greedy_rev:.2f} / 15-min window')
print(f'  Our System (WSM)         : ${our_rev:.2f} / 15-min window')
print('-'*55)
print(f'  Improvement vs Random    : {((our_rev-random_rev)/random_rev)*100:.1f}%')
print(f'  Improvement vs Nearest   : {((our_rev-nearest_rev)/nearest_rev)*100:.1f}%')
print(f'  Improvement vs Greedy    : {((our_rev-greedy_rev)/greedy_rev)*100:.1f}%')
print('='*55)

       BASELINE COMPARISON (Per-Driver Revenue)
  Random Zone              : $16.67 / 15-min window
  Nearest Zone             : $41.70 / 15-min window
  Greedy Demand            : $11.85 / 15-min window
  Our System (WSM)         : $52.89 / 15-min window
-------------------------------------------------------
  Improvement vs Random    : 217.3%
  Improvement vs Nearest   : 26.8%
  Improvement vs Greedy    : 346.3%


## 12. Graphs

In [26]:
# ── Graph 1: Actual vs Predicted ──────────────────────────────────────────────
sample_idx = np.random.choice(len(y_test), min(3000, len(y_test)), replace=False)
y_t_s = y_test.values[sample_idx]
y_p_s = y_pred[sample_idx]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_t_s, y_p_s, alpha=0.3, s=12, color='steelblue', edgecolors='none')
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual Demand')
ax.set_ylabel('Predicted Demand')
ax.set_title(f'Actual vs Predicted Demand\n$R^2$={r2:.4f}, MAPE={mape:.1f}%')
ax.legend()
plt.tight_layout()
plt.savefig('graph1_actual_vs_predicted.png', dpi=150)
plt.show()
print('Graph 1 saved')

In [27]:
# ── Graph 2: Residual Distribution (Reviewer Fix #15) ─────────────────────────
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals, bins=60, color='steelblue', edgecolor='black', linewidth=0.3)
axes[0].axvline(0, color='red', linestyle='--', lw=1.5)
axes[0].set_xlabel('Residual (Actual − Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution')

axes[1].scatter(y_pred[sample_idx], residuals[sample_idx],
                alpha=0.3, s=10, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted Demand')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual vs Predicted')

plt.tight_layout()
plt.savefig('graph2_residuals.png', dpi=150)
plt.show()
print('Graph 2 saved')

Graph 2 saved


In [28]:
# ── Graph 3: Feature Importance ───────────────────────────────────────────────
imp_df = pd.DataFrame({
    'feature':    features,
    'importance': model.feature_importances_
}).sort_values('importance')

plt.figure(figsize=(7, 5))
plt.barh(imp_df['feature'], imp_df['importance'],
         color='steelblue', edgecolor='black', linewidth=0.4)
plt.xlabel('Importance Score')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('graph3_feature_importance.png', dpi=150)
plt.show()
print('Graph 3 saved')

Graph 3 saved


In [29]:
# ── Graph 4: WSM Score — Top 20 Zones ────────────────────────────────────────
top20 = zones_sorted.head(20)
colors = ['green' if i == 0 else 'steelblue' for i in range(len(top20))]

plt.figure(figsize=(10, 5))
plt.bar(top20['PULocationID'].astype(str), top20['WSM_score'],
        color=colors, edgecolor='black', linewidth=0.4)
plt.xlabel('Zone (PULocationID)')
plt.ylabel('WSM Score')
plt.title('WSM Welfare Score — Top 20 Zones\n(Green = Recommended)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('graph4_wsm_scores.png', dpi=150)
plt.show()
print('Graph 4 saved')

Graph 4 saved


In [30]:
# ── Graph 5: Baseline Comparison Bar Chart ────────────────────────────────────
labels   = ['Random\nZone', 'Nearest\nZone', 'Greedy\nDemand', 'Our System\n(WSM)']
revenues = [random_rev, nearest_rev, greedy_rev, our_rev]
colors5  = ['tomato', 'orange', 'goldenrod', 'green']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, revenues, color=colors5, edgecolor='black', linewidth=0.5, width=0.5)
for bar, val in zip(bars, revenues):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(revenues)*0.01,
            f'${val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Avg Per-Driver Revenue / 15-min Window ($)')
ax.set_title('Baseline Comparison\n(Corrected Per-Driver Earnings)')
plt.tight_layout()
plt.savefig('graph5_baseline_comparison.png', dpi=150)
plt.show()
print('Graph 5 saved')

Graph 5 saved


In [31]:
# ── Graph 6: Zone Distribution Histogram ─────────────────────────────────────
plt.figure(figsize=(10, 4))
zone_counts = df['PULocationID'].value_counts()
zone_counts.head(40).plot(kind='bar', color='steelblue',
                           edgecolor='black', linewidth=0.3)
plt.xlabel('PULocationID (Top 40 Zones)')
plt.ylabel('Trip Count')
plt.title(f'Zone Trip Distribution (Gini={gini(zone_counts):.3f})')
plt.xticks(fontsize=7, rotation=90)
plt.tight_layout()
plt.savefig('graph6_zone_distribution.png', dpi=150)
plt.show()
print('Graph 6 saved')

Graph 6 saved


In [32]:
# ── Graph 7: Revenue vs Idle Penalty per Top 15 Zones ────────────────────────
top15 = zones_sorted.head(15).copy()
top15['revenue_score'] = (
    top15['predicted_demand'] * top15['avg_fare'] / top15['avg_drivers']
)
top15['idle_score'] = beta / (top15['demand_density'] + 1e-6)

x     = np.arange(len(top15))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, top15['revenue_score'], width,
       label='Per-Driver Revenue (α×R)', color='green', alpha=0.82,
       edgecolor='black', linewidth=0.4)
ax.bar(x + width/2, top15['idle_score'], width,
       label='Idle Penalty (β×I)', color='red', alpha=0.82,
       edgecolor='black', linewidth=0.4)
ax.set_xticks(x)
ax.set_xticklabels(top15['PULocationID'].astype(int), rotation=45, fontsize=8)
ax.set_xlabel('Zone (PULocationID)')
ax.set_ylabel('Score Component Value')
ax.set_title('Per-Driver Revenue vs Idle Penalty — Top 15 Zones')
ax.legend()
plt.tight_layout()
plt.savefig('graph7_revenue_vs_idle.png', dpi=150)
plt.show()
print('Graph 7 saved')

Graph 7 saved


In [33]:
# ── Summary printout ──────────────────────────────────────────────────────────
print('\n' + '='*55)
print('           FINAL SUMMARY')
print('='*55)
print(f'  Dataset          : Jan–Mar 2025 NYC Yellow Taxi')
print(f'  Zones used       : {len(zones)} (official NYC 263-zone system)')
print(f'  R²               : {r2:.4f}')
print(f'  MAE              : {mae:.4f}')
print(f'  RMSE             : {rmse:.4f}')
print(f'  MAPE             : {mape:.2f}%')
print(f'  NRMSE            : {nrmse:.2f}%')
print(f'  Best zone        : Zone {int(best_zone["PULocationID"])}')
print(f'  WSM weights      : α={alpha}, β={beta}, γ={gamma}')
print(f'  Per-driver rev   : ${our_rev:.2f} / 15-min window')
print(f'  vs Random        : +{((our_rev-random_rev)/random_rev)*100:.1f}%')
print(f'  vs Nearest       : +{((our_rev-nearest_rev)/nearest_rev)*100:.1f}%')
print(f'  vs Greedy        : +{((our_rev-greedy_rev)/greedy_rev)*100:.1f}%')
print('='*55)


           FINAL SUMMARY
  Dataset          : Jan–Mar 2025 NYC Yellow Taxi
  Zones used       : 223 (official NYC 263-zone system)
  R²               : 0.9405
  MAE              : 2.5953
  RMSE             : 5.0695
  MAPE             : 40.97%
  NRMSE            : 1.96%
  Best zone        : Zone 132
  WSM weights      : α=0.6, β=0.2, γ=0.1
  Per-driver rev   : $52.89 / 15-min window
  vs Random        : +217.3%
  vs Nearest       : +26.8%
  vs Greedy        : +346.3%
